In [2]:
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from cryptography.hazmat.primitives import serialization

predictions_df = pd.read_csv('inference_predictions.csv')
predictions_df.columns = [c.upper() for c in predictions_df.columns]

with open(r"C:\Users\reddy\OneDrive\Desktop\cat-fleet-predictive-maintenance\rsa_key.p8", "rb") as key_file:
    private_key = serialization.load_pem_private_key(key_file.read(), password=None)
pkb = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

sf_conn = snowflake.connector.connect(
    account='TGB56921',
    user='SPT',
    private_key=pkb,
    warehouse='SPT_WH',
    database='CAT_FLEET_DB',
    schema='PUBLIC'
)

success, num_chunks, num_rows, output = write_pandas(
    conn=sf_conn,
    df=predictions_df,
    table_name='FLEET_PREDICTIONS',
    auto_create_table=True,
    overwrite=True
)
print(f"Upload success: {success}, Rows written: {num_rows}")
sf_conn.close()

Upload success: True, Rows written: 250
